In [1]:
from pyspark.sql import SparkSession

# Test session creation
spark = SparkSession.builder \
    .appName("Test_Session") \
    .getOrCreate()

print("Success! Spark version running:", spark.version)

C:\Users\izhan\anaconda3\envs\spark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Success! Spark version running: 4.2.0


In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as spark_sum, to_timestamp, min, max, mean

# Initialize the Spark Session
spark = SparkSession.builder \
    .appName("Ecommerce_Week5_Analysis") \
    .getOrCreate()

# Define the folder path
folder_path = r"D:\CELEBAL_CEI_IZHAN\WEEK5_Celebal\ecommerce dataset"

# Automatically find the CSV file in that folder
files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
if not files:
    raise FileNotFoundError(f"No CSV file found in {folder_path}.")

file_name = files[0]
path_to_file = os.path.join(folder_path, file_name)

# Load the dataset
df = spark.read.csv(path_to_file, header=True, inferSchema=True)
print(f"Loaded: {file_name} successfully!")
df.show(3)

Loaded: Ecommerce.csv successfully!
+-----------+----------+----------+-----------+---------+-----------------+----------+----------------+----------+--------+----------------+---------------+-------+------------+----------------+-------------+---------+--------------+------+-----------+--------------------+--------------+---------+-----------+-------------+------------+-----------------------+------------------+--------+
|customer_id|session_id|visit_date|device_type|user_type|marketing_channel|product_id|product_category|unit_price|quantity|discount_percent|discount_amount|revenue|pages_viewed|time_on_site_sec|added_to_cart|purchased|cart_abandoned|rating|review_text|review_helpful_votes|payment_method|visit_day|visit_month|visit_weekday|visit_season|session_duration_bucket|revenue_normalized|location|
+-----------+----------+----------+-----------+---------+-----------------+----------+----------------+----------+--------+----------------+---------------+-------+------------+-------

In [9]:
# Q4: Given a DataFrame df_sales, write a query to filter for rows where 

from pyspark.sql.types import StringType

q4_result = (df.filter(col("location").cast(StringType()) == "West") 
             .groupBy("product_category")
             .agg(avg("revenue").alias("avg_sale_amount")))

q4_result.show(5)

+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
+----------------+---------------+



In [7]:
# Q5: What is the difference between .na.drop() and .na.fill()? Provide a code 

df_filled_status = df.na.fill({"payment_method": "Unknown"})
df_filled_status.select("payment_method").distinct().show(5)

+--------------+
|payment_method|
+--------------+
|             1|
|             3|
|             5|
|             4|
|             2|
+--------------+
only showing top 5 rows


In [10]:
# Q6: Write a query to find the total count of records for each city in a DataFrame

q6_result = (df.groupBy("location")
             .agg(count("*").alias("record_count"))
             .filter(col("record_count") > 100))

q6_result.show()

+--------+------------+
|location|record_count|
+--------+------------+
|     148|         104|
|      31|         121|
|      85|         109|
|     137|         113|
|      65|         105|
|      53|         103|
|     133|         113|
|      78|         114|
|     108|         102|
|     211|         124|
|      34|         111|
|     193|         127|
|     126|         115|
|     101|         102|
|      81|         127|
|     210|         103|
|     183|         139|
|      28|         115|
|      76|         117|
|      27|         112|
+--------+------------+
only showing top 20 rows


In [11]:
# Q7: How does the immutability of Spark DataFrames affect how you perform

df_cleaned = df.drop("session_id").withColumnRenamed("revenue", "total_price")
df_cleaned.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- device_type: integer (nullable = true)
 |-- user_type: integer (nullable = true)
 |-- marketing_channel: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_category: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- total_price: double (nullable = true)
 |-- pages_viewed: integer (nullable = true)
 |-- time_on_site_sec: integer (nullable = true)
 |-- added_to_cart: integer (nullable = true)
 |-- purchased: integer (nullable = true)
 |-- cart_abandoned: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review_text: integer (nullable = true)
 |-- review_helpful_votes: integer (nullable = true)
 |-- payment_method: integer (nullable = true)
 |-- visit_day: integer (nullable = true)
 |-- 

In [12]:
# Q10: Write the code to revise a column named raw_timestamp by casting 

df_timestamped = df.withColumn("event_time", to_timestamp(col("visit_date"), "yyyy-MM-dd"))
df_timestamped.select("visit_date", "event_time").printSchema()

root
 |-- visit_date: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [13]:
# Q12: Write a code snippet that identifies and removes rows where the email 
# column contains null values OR the username is an empty string.

df_valid_users = df.filter(col("customer_id").isNotNull() & (col("payment_method") != ""))
print("Cleaned rows syntax applied successfully.")

Cleaned rows syntax applied successfully.


In [14]:
# Q13: How do you use the .agg() function to calculate multiple statistics 

q13_result = df.agg(
    min("unit_price").alias("min_price"),
    max("unit_price").alias("max_price"),
    mean("unit_price").alias("avg_price")
)

q13_result.show()

+---------+---------+----------------+
|min_price|max_price|       avg_price|
+---------+---------+----------------+
|    50.05|  1999.83|782.319010400002|
+---------+---------+----------------+



In [15]:
# Q15: Write a final processing pipeline that:
# 1. Filters out duplicates.
# 2. Fills null prices with 0.
# 3. Groups by store_id to calculate total revenue.

final_pipeline = (df
    .dropDuplicates()
    .na.fill(0, subset=["unit_price", "revenue"])
    .groupBy("customer_id")
    .agg(spark_sum("revenue").alias("total_revenue"))
)

final_pipeline.show(10)

+-----------+-----------------+
|customer_id|    total_revenue|
+-----------+-----------------+
|       6466|              0.0|
|       5803|              0.0|
|       8638|          1292.67|
|       7754|              0.0|
|       1645|          3067.74|
|       4900|          1777.66|
|       7253|              0.0|
|       6658|4624.110000000001|
|       9900|              0.0|
|       3997|              0.0|
+-----------+-----------------+
only showing top 10 rows
